# 🍯 BiteVision — Human & Animal Detection
## TensorFlow Lite Model Training Pipeline

This notebook trains a **MobileNetV2** model to detect humans and animals, then exports it as a **TensorFlow Lite** model for use in the BiteVision Android app.

### Pipeline Overview:
1. Install dependencies
2. Download & prepare dataset (COCO subset)
3. Build MobileNetV2 feature extractor
4. Train classification head
5. Evaluate model
6. Convert to TFLite (quantized)
7. Add TFLite metadata
8. Export labels.txt

---
## 1. Install Dependencies

In [ ]:
!pip install tensorflow==2.14.0 tensorflow-hub tensorflow-datasets tflite-support matplotlib numpy pillow seaborn scikit-learn -q
print('✅ Dependencies installed')

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, shutil
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

tf.random.set_seed(42)
np.random.seed(42)

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 20
NUM_CLASSES = 3
CLASS_NAMES = ['human', 'animal', 'other']
BASE_DIR    = Path('./bite_vision_data')
MODEL_DIR   = Path('./bite_vision_model')

BASE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
print('✅ Setup complete')

---
## 2. Dataset Preparation

We use **TensorFlow Datasets** (TFDS) to load COCO and map to 3 categories.

In [ ]:
import tensorflow_datasets as tfds

print('Loading COCO 2017 dataset (this may take a few minutes)...')
(ds_train_raw, ds_val_raw), ds_info = tfds.load(
    'coco/2017',
    split=['train[:20%]', 'validation[:30%]'],
    with_info=True,
    as_supervised=False,
)
print(f'Train: {ds_train_raw.cardinality().numpy()} | Val: {ds_val_raw.cardinality().numpy()}')

In [ ]:
# COCO class indices (0-based in TFDS):
# 0=person, 14=bird, 15=cat, 16=dog, 17=horse, 18=sheep,
# 19=cow, 20=elephant, 21=bear, 22=zebra, 23=giraffe
HUMAN_IDX  = tf.constant([0], dtype=tf.int64)
ANIMAL_IDX = tf.constant([14,15,16,17,18,19,20,21,22,23], dtype=tf.int64)

def preprocess(sample):
    image = tf.image.resize(sample['image'], IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    label_ids = tf.cast(sample['objects']['label'], tf.int64)
    is_human  = tf.reduce_any(tf.reduce_any(tf.equal(tf.expand_dims(label_ids,1), tf.reshape(HUMAN_IDX,[1,-1])),axis=1))
    is_animal = tf.reduce_any(tf.reduce_any(tf.equal(tf.expand_dims(label_ids,1), tf.reshape(ANIMAL_IDX,[1,-1])),axis=1))
    label = tf.cond(is_human, lambda: tf.constant(0), lambda: tf.cond(is_animal, lambda: tf.constant(1), lambda: tf.constant(2)))
    return image, tf.one_hot(label, NUM_CLASSES)

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return tf.clip_by_value(image, 0.0, 1.0), label

AUTOTUNE = tf.data.AUTOTUNE
ds_train = ds_train_raw.map(preprocess, num_parallel_calls=AUTOTUNE).cache().shuffle(1000).map(augment, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val   = ds_val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
print('✅ Datasets ready')

---
## 3. Build MobileNetV2 Model

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3), name='input_image')
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = tf.keras.Model(inputs, outputs, name='BiteVision')
model.summary()
print(f'Trainable params: {sum([np.prod(v.shape) for v in model.trainable_variables]):,}')

---
## 4. Training

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint(str(MODEL_DIR/'best_model.keras'), monitor='val_accuracy', save_best_only=True),
]

# Phase 1: Head only
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
print('🚀 Phase 1: Training head...')
h1 = model.fit(ds_train, validation_data=ds_val, epochs=10, callbacks=callbacks)

# Phase 2: Fine-tune
base_model.trainable = True
for layer in base_model.layers[:-30]: layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
print('🔓 Phase 2: Fine-tuning...')
h2 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, callbacks=callbacks, initial_epoch=len(h1.epoch))

---
## 5. Evaluation

In [ ]:
loss, acc = model.evaluate(ds_val)
print(f'Val Loss: {loss:.4f} | Val Accuracy: {acc:.4f} ({acc*100:.1f}%)')

y_true, y_pred = [], []
for imgs, lbls in ds_val:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(tf.argmax(lbls, axis=1).numpy())
    y_pred.extend(tf.argmax(preds, axis=1).numpy())

fig, ax = plt.subplots(figsize=(8,6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Confusion Matrix', fontweight='bold'); ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.savefig(MODEL_DIR/'confusion_matrix.png', dpi=150); plt.show()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

---
## 6. Convert to TFLite (INT8 Quantized)

In [ ]:
model.save(MODEL_DIR/'bite_vision.keras')
print('✅ Keras model saved')

def rep_dataset():
    for imgs, _ in ds_val.unbatch().batch(1).take(100):
        yield [tf.cast(imgs, tf.float32)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_dataset
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type  = tf.uint8
conv.inference_output_type = tf.uint8

tflite_model = conv.convert()
tflite_path = MODEL_DIR/'bite_vision_quant.tflite'
tflite_path.write_bytes(tflite_model)
print(f'✅ TFLite model: {len(tflite_model)/1024:.1f} KB → {tflite_path}')

---
## 7. Export Labels & Test Inference

In [ ]:
labels_path = MODEL_DIR/'labels.txt'
labels_path.write_text('\n'.join(CLASS_NAMES))
print(f'✅ Labels: {labels_path.read_text()}')

# Quick TFLite inference test
interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
inp  = interp.get_input_details()
outp = interp.get_output_details()
print(f'Input:  {inp[0]["shape"]} {inp[0]["dtype"]}')
print(f'Output: {outp[0]["shape"]} {outp[0]["dtype"]}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('TFLite Predictions', fontsize=14, fontweight='bold')
COLORS = ['#3B82F6','#10B981','#8B5CF6']
EMOJIS = ['👤','🐾','🔍']

for (imgs, lbls), ax in zip(ds_val.unbatch().take(8), axes.flat):
    img_arr = (imgs.numpy()*255).astype(np.uint8)[np.newaxis]
    interp.set_tensor(inp[0]['index'], img_arr)
    interp.invoke()
    out = interp.get_tensor(outp[0]['index'])[0].astype(np.float32)/255.0
    pred = np.argmax(out)
    true = tf.argmax(lbls).numpy()
    ax.imshow(imgs.numpy())
    ax.set_title(f'{EMOJIS[pred]} {CLASS_NAMES[pred].upper()} {out[pred]*100:.0f}%', color=COLORS[pred], fontweight='bold', fontsize=9)
    for s in ax.spines.values(): s.set_edgecolor('#10B981' if pred==true else '#EF4444'); s.set_linewidth(3)
    ax.axis('off')

plt.tight_layout()
plt.savefig(MODEL_DIR/'tflite_predictions.png', dpi=150)
plt.show()
print('✅ Done! Copy bite_vision_quant.tflite → app/src/main/assets/detect.tflite')